In [18]:
import os
from typing import Tuple
from pathlib import Path
import numpy as np
import librosa
import soundfile as sf

FAKE_IN_DIR = Path("../archive/fake_songs")
REAL_IN_DIR = Path("../archive/real_songs")
FAKE_OUT_DIR = Path("../crop_data/fake_6s")
REAL_OUT_DIR = Path("../crop_data/real_6s")
IN_DIR = FAKE_IN_DIR
OUT_DIR = FAKE_OUT_DIR

In [19]:
def select_richest_6s(
    y: np.ndarray,
    sr: int = 16000,
    win_sec: float = 6.0,
    hop_sec: float = 1.0,
) -> Tuple[int, np.ndarray]:
    """
    Fast version: select the most information-rich fixed-length segment
    from a mono waveform.

    This computes STFT/RMS once for the full audio, then aggregates frame-level
    features over 6-second candidate windows with 1-second stride.

    Metrics:
    - RMS energy
    - spectral flux / onset strength
    - spectral entropy
    - spectral bandwidth
    - silence ratio

    Final score after safe z-score normalization:
        0.30 * rms
      + 0.25 * flux
      + 0.20 * entropy
      + 0.15 * bandwidth
      - 0.50 * silence_ratio

    Parameters
    ----------
    y:
        Mono waveform, 1D numpy array.
    sr:
        Sampling rate.
    win_sec:
        Segment length in seconds.
    hop_sec:
        Candidate window stride in seconds.

    Returns
    -------
    best_start_sample:
        Start sample of the selected segment.
    best_segment:
        Segment with exactly int(win_sec * sr) samples.
    """
    if y is None:
        raise ValueError("Input waveform y must not be None.")

    y = np.asarray(y)

    if y.size == 0:
        raise ValueError("Input waveform y is empty.")

    if y.ndim != 1:
        raise ValueError(f"Input waveform must be mono/1D, got shape {y.shape}.")

    if sr <= 0:
        raise ValueError("sr must be positive.")

    if win_sec <= 0:
        raise ValueError("win_sec must be positive.")

    if hop_sec <= 0:
        raise ValueError("hop_sec must be positive.")

    y = y.astype(np.float32, copy=False)
    y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0)

    n_fft = 2048
    stft_hop = 512

    win_len = int(win_sec * sr)
    scan_hop = int(hop_sec * sr)

    if win_len <= 0 or scan_hop <= 0:
        raise ValueError("Invalid win_sec or hop_sec.")

    if len(y) < win_len:
        padded = np.zeros(win_len, dtype=np.float32)
        padded[:len(y)] = y
        return 0, padded

    # Candidate start samples
    starts = np.arange(0, len(y) - win_len + 1, scan_hop, dtype=np.int64)

    # Compute full-song RMS once
    global_rms = librosa.feature.rms(
        y=y,
        frame_length=n_fft,
        hop_length=stft_hop,
        center=True,
    )[0].astype(np.float64)

    silence_threshold = 0.1 * float(np.median(global_rms))

    # Compute full-song STFT once
    mag = np.abs(
        librosa.stft(
            y,
            n_fft=n_fft,
            hop_length=stft_hop,
            center=True,
        )
    ).astype(np.float32)

    n_bins, n_frames = mag.shape
    eps = 1e-10

    # Frame sample positions
    frame_samples = librosa.frames_to_samples(
        np.arange(n_frames),
        hop_length=stft_hop,
    )

    left = np.searchsorted(frame_samples, starts, side="left")
    right = np.searchsorted(frame_samples, starts + win_len, side="right")

    left = np.clip(left, 0, n_frames - 1)
    right = np.clip(right, left + 1, n_frames)

    def range_mean(values: np.ndarray, lidx: np.ndarray, ridx: np.ndarray) -> np.ndarray:
        values = np.asarray(values, dtype=np.float64)
        csum = np.concatenate([[0.0], np.cumsum(values)])
        denom = np.maximum(ridx - lidx, 1)
        return (csum[ridx] - csum[lidx]) / denom

    def safe_zscore(values: np.ndarray, eps_std: float = 1e-8) -> np.ndarray:
        values = np.asarray(values, dtype=np.float64)
        std = float(np.std(values))
        if std < eps_std:
            return np.zeros_like(values, dtype=np.float64)
        return (values - float(np.mean(values))) / std

    # 1. RMS energy per frame
    rms_frame = global_rms

    # 2. Spectral flux per frame
    flux_frame = np.zeros(n_frames, dtype=np.float64)
    if n_frames > 1:
        diff = np.diff(mag, axis=1)
        pos_diff = np.maximum(diff, 0.0)
        flux_frame[1:] = np.sqrt(np.sum(pos_diff * pos_diff, axis=0))

    # 3. Spectral entropy per frame
    power = mag.astype(np.float64) ** 2
    power_sum = np.sum(power, axis=0, keepdims=True)
    prob = power / (power_sum + eps)
    entropy_frame = -np.sum(prob * np.log(prob + eps), axis=0) / np.log(n_bins + eps)

    # 4. Spectral bandwidth per frame, manual version for speed
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft).astype(np.float64)[:, None]
    mag64 = mag.astype(np.float64)
    mag_sum = np.sum(mag64, axis=0, keepdims=True) + eps
    centroid = np.sum(freqs * mag64, axis=0, keepdims=True) / mag_sum
    bandwidth_frame = np.sqrt(
        np.sum(((freqs - centroid) ** 2) * mag64, axis=0) / mag_sum.squeeze()
    )

    # 5. Silence ratio per frame
    silence_frame = (global_rms < silence_threshold).astype(np.float64)

    rms_scores = range_mean(rms_frame, left, right)
    flux_scores = range_mean(flux_frame, left, right)
    entropy_scores = range_mean(entropy_frame, left, right)
    bandwidth_scores = range_mean(bandwidth_frame, left, right)
    silence_ratios = range_mean(silence_frame, left, right)

    rms_z = safe_zscore(rms_scores)
    flux_z = safe_zscore(flux_scores)
    entropy_z = safe_zscore(entropy_scores)
    bandwidth_z = safe_zscore(bandwidth_scores)
    silence_z = safe_zscore(silence_ratios)

    final_scores = (
        0.30 * rms_z
        + 0.25 * flux_z
        + 0.20 * entropy_z
        + 0.15 * bandwidth_z
        - 0.50 * silence_z
    )

    best_idx = int(np.argmax(final_scores))
    best_start_sample = int(starts[best_idx])

    best_segment = y[best_start_sample : best_start_sample + win_len]

    if len(best_segment) < win_len:
        padded = np.zeros(win_len, dtype=np.float32)
        padded[:len(best_segment)] = best_segment
        best_segment = padded
    else:
        best_segment = best_segment.astype(np.float32, copy=False)

    return best_start_sample, best_segment

In [21]:
from pathlib import Path
import librosa

audio_files = sorted(IN_DIR.glob("*.mp3"))

path = str(audio_files[0])
print("Using:", path)

y, sr = librosa.load(path, sr=16000, mono=True)

best_start_sample, best_segment = select_richest_6s(
    y=y,
    sr=sr,
    win_sec=6.0,
    hop_sec=1.0,
)

best_start_sec = best_start_sample / sr

print("best_start_sample:", best_start_sample)
print("best_start_sec:", best_start_sec)
print("segment shape:", best_segment.shape)
print("segment duration:", len(best_segment) / sr, "seconds")

Using: ../archive/fake_songs/fake_00001_suno_0.mp3
best_start_sample: 1936000
best_start_sec: 121.0
segment shape: (96000,)
segment duration: 6.0 seconds


In [22]:
# Colab free thường 2 CPU cores.
# Nếu Colab Pro / máy nhiều core, thử 4 hoặc 6.
MAX_WORKERS = min(18, os.cpu_count() or 2)

pending_files = [
    str(p) for p in audio_files
    if not (OUT_DIR / f"{p.stem}.mp3").exists()
    or (OUT_DIR / f"{p.stem}.mp3").stat().st_size <= 1000
]

print("CPU cores:", os.cpu_count())
print("Workers:", MAX_WORKERS)
print("Pending:", len(pending_files))

CPU cores: 24
Workers: 18
Pending: 49074


In [23]:
from pathlib import Path
import subprocess
import tempfile
import csv

import librosa
import soundfile as sf
from tqdm import tqdm


OUT_DIR.mkdir(parents=True, exist_ok=True)

SR = 16000
BITRATE = "128k"

audio_files = sorted(
    list(IN_DIR.glob("*.mp3")) +
    list(IN_DIR.glob("*.wav")) +
    list(IN_DIR.glob("*.flac")) +
    list(IN_DIR.glob("*.m4a"))
)

print("Found audio files:", len(audio_files))
print("Output folder:", OUT_DIR)

Found audio files: 49074
Output folder: ../crop_data/fake_6s


In [24]:
def save_segment_as_mp3(segment, sr: int, out_path: Path, bitrate: str = "128k") -> None:
    """
    Save a mono waveform segment to MP3 using a temporary WAV file and ffmpeg.
    """
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        tmp_wav = Path(tmp.name)

    try:
        sf.write(tmp_wav, segment, sr)

        cmd = [
            "ffmpeg",
            "-y",
            "-loglevel", "error",
            "-i", str(tmp_wav),
            "-codec:a", "libmp3lame",
            "-b:a", bitrate,
            str(out_path),
        ]

        subprocess.run(cmd, check=True)

    finally:
        if tmp_wav.exists():
            tmp_wav.unlink()

In [25]:
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
import os
import subprocess
import tempfile
import csv

import librosa
import soundfile as sf
from tqdm import tqdm

SR = 16000
BITRATE = "128k"

audio_files = sorted(IN_DIR.glob("*.mp3"))

print("Total input:", len(audio_files))
print("Already cropped:", len(list(OUT_DIR.glob("*.mp3"))))

Total input: 49074
Already cropped: 0


In [27]:
def save_segment_as_mp3(segment, sr: int, out_path: str, bitrate: str = "128k") -> None:
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        tmp_wav = tmp.name

    try:
        sf.write(tmp_wav, segment, sr)

        cmd = [
            "ffmpeg",
            "-y",
            "-loglevel", "error",
            "-i", tmp_wav,
            "-codec:a", "libmp3lame",
            "-b:a", bitrate,
            out_path,
        ]

        subprocess.run(cmd, check=True)

    finally:
        if os.path.exists(tmp_wav):
            os.remove(tmp_wav)


def process_one(path_str: str):
    path = Path(path_str)
    out_path = OUT_DIR / f"{path.stem}.mp3"

    # resume
    if out_path.exists() and out_path.stat().st_size > 1000:
        return {
            "status": "skipped",
            "source_file": path.name,
            "crop_file": out_path.name,
            "source_path": str(path),
            "crop_path": str(out_path),
            "best_start_sec": None,
            "fake_type": "Real",
            "label": 1,
            "error": "",
        }

    try:
        y, sr = librosa.load(path, sr=SR, mono=True)

        best_start_sample, best_segment = select_richest_6s(
            y=y,
            sr=sr,
            win_sec=6.0,
            hop_sec=1.0,
        )

        best_start_sec = best_start_sample / sr

        save_segment_as_mp3(
            segment=best_segment,
            sr=sr,
            out_path=str(out_path),
            bitrate=BITRATE,
        )

        return {
            "status": "ok",
            "source_file": path.name,
            "crop_file": out_path.name,
            "source_path": str(path),
            "crop_path": str(out_path),
            "best_start_sec": best_start_sec,
            "fake_type": "Real",
            "label": 1,
            "error": "",
        }

    except Exception as e:
        return {
            "status": "error",
            "source_file": path.name,
            "crop_file": "",
            "source_path": str(path),
            "crop_path": "",
            "best_start_sec": "",
            "fake_type": "Real",
            "label": 1,
            "error": repr(e),
        }

In [28]:
rows = []
errors = []

with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(process_one, p) for p in pending_files]

    for fut in tqdm(as_completed(futures), total=len(futures)):
        r = fut.result()

        if r["status"] == "error":
            errors.append(r)
        else:
            rows.append(r)

manifest_path = OUT_DIR / "real_crop_manifest.csv"
error_path = OUT_DIR / "real_crop_errors.csv"

with open(manifest_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "status",
            "source_file",
            "crop_file",
            "source_path",
            "crop_path",
            "best_start_sec",
            "fake_type",
            "label",
            "error",
        ],
    )
    writer.writeheader()
    writer.writerows(rows)

if errors:
    with open(error_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "status",
                "source_file",
                "crop_file",
                "source_path",
                "crop_path",
                "best_start_sec",
                "fake_type",
                "label",
                "error",
            ],
        )
        writer.writeheader()
        writer.writerows(errors)

print("Done.")
print("Saved mp3:", len(list(OUT_DIR.glob('*.mp3'))))
print("Errors:", len(errors))
print("Manifest:", manifest_path)

100%|██████████| 49074/49074 [50:17<00:00, 16.26it/s]  


Done.
Saved mp3: 49074
Errors: 0
Manifest: ../crop_data/fake_6s/real_crop_manifest.csv
